<a href="https://colab.research.google.com/github/seirah-yang/BootCamp/blob/main/LLM_%EA%B8%B0%EB%B0%98%EC%9D%98_%EB%A6%AC%EB%9E%AD%ED%82%B9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<기본 RAG> - 벡터 검색으로 문서 4개를 찾아옵니다
- LLM 호출 1회: 검색된 4개 문서 전체를 바탕으로 답변을 생성합니다
- 총 LLM 호출: 1회

<리랭킹 적용 RAG> - 벡터 검색으로 문서 4개를 찾아옵니다
- LLM 호출 4회: 각 문서와 질문의 관련성을 1-10점 사이로 평가합니다
- LLM 호출 1회: 관련성 점수가 높은 상위 2개 문서만으로 답변을 생성합니다
- 총 LLM 호출: 5회

In [ ]:
!pip install langchain_openai langchain_community langchain_chroma pypdf

In [3]:
import os
import urllib.request
import json
from typing import List
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain.schema import Document
import requests

In [4]:
# 분석할 PDF 파일을 웹에서 다운로드.
url = "https://github.com/llama-index-tutorial/llama-index-tutorial/raw/main/ch07/2023_%EB%B6%81%ED%95%9C%EC%9D%B8%EA%B6%8C%EB%B3%B4%EA%B3%A0%EC%84%9C.pdf"
filename = "2023_북한인권보고서.pdf"

response = requests.get(url)
with open(filename, "wb") as f:
    f.write(response.content)

print(f"{filename} 다운로드 완료")

2023_북한인권보고서.pdf 다운로드 완료


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "skt/A.X-4.0"
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
model.eval()
tokenizer = AutoTokenizer.from_pretrained(model_name)

messages = [
    {"role": "system", "content": "당신은 사용자가 제공하는 영어 문장들을 한국어로 번역하는 AI 전문가입니다."},
    {"role": "user", "content": "The first human went into space and orbited the Earth on April 12, 1961."},
]
input_ids = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt").to(model.device)

with torch.no_grad():
    output = model.generate(
        input_ids,
        max_new_tokens=128,
        do_sample=False,
    )

len_input_prompt = len(input_ids[0])
response = tokenizer.decode(output[0][len_input_prompt:], skip_special_tokens=True)
print(response)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!
The explicitly set RoPE scaling factor (config.rope_scaling['factor'] = 2.0) does not match the ratio implicitly set by other parameters (implicit factor = post-yarn context length / pre-yarn context length = config.max_position_embeddings / config.rope_scaling['original_max_position_embeddings'] = 1.0). Using the explicit factor (2.0) in YaRN. This may cause unexpected behaviour in model usage, please correct the 'max_position_embeddings' fields in the model config.


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

...

model-00008-of-00030.safetensors:   0%|          | 0.00/4.96G [00:00<?, ?B/s]

model-00007-of-00030.safetensors:   0%|          | 0.00/4.78G [00:00<?, ?B/s]

model-00001-of-00030.safetensors:   0%|          | 0.00/4.70G [00:00<?, ?B/s]

model-00006-of-00030.safetensors:   0%|          | 0.00/4.78G [00:00<?, ?B/s]

model-00005-of-00030.safetensors:   0%|          | 0.00/4.78G [00:00<?, ?B/s]

In [ ]:
from openai import OpenAI
def call(messages, model):
    completion = client.chat.completions.create(
        model=model,
        messages=messages,
    )
    print(completion.choices[0].message)

client = OpenAI(
    base_url="http://localhost:8000/v1",
    api_key="api_key"
)
model = "skt/A.X-4.0"
messages = [{"role": "user", "content": "content"}]
call(messages, model)
embed_model = OpenAIEmbeddings(model="text-embedding-3-large")  # 임베딩 모델 사용

# 문서 분할 설정
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50,
)

# PDF 문서를 읽고 벡터 인덱스 생성
loader = PyPDFLoader("2023_북한인권보고서.pdf")  # PDF 문서 로더
documents = loader.load()  # 문서에서 텍스트 추출
chunks = text_splitter.split_documents(documents)  # 문서 분할
vector_store = Chroma.from_documents(chunks, embed_model)  # 추출된 텍스트로 벡터 인덱스 생성


In [8]:
class DocumentScorer:
    # LLM을 사용해 문서의 관련성을 정밀하게 평가하고 점수를 매기는 클래스

    def __init__(self, llm):
        self.llm = llm

    def evaluate_document(self, query: str, content: str) -> float:
        # LLM을 사용해 문서와 쿼리 간의 의미적 관련성을 1-10점으로 평가
        prompt = f"""
        아래 주어진 질문과 문서의 관련성을 평가해주세요.

        [평가 기준]
        - 문서가 질문에서 요구하는 정보를 직접적으로 포함하면 8-10점
        - 문서가 질문과 관련된 맥락을 포함하지만 직접적인 답이 아니면 4-7점
        - 문서가 질문과 거의 관련이 없으면 1-3점

        [주의사항]
        - 단순히 비슷한 단어가 등장하는 것은 높은 점수의 근거가 될 수 없습니다
        - 질문의 의도와 문맥을 정확히 파악하여 평가해주세요
        - 시간, 장소, 수치 등 구체적인 정보의 일치 여부를 중요하게 고려해주세요

        질문: {query}
        문서: {content}

        응답은 반드시 다음 JSON 형식이어야 합니다. 백틱은 쓰지마십시오.:
        {{"relevance_score": float}}
        """

        try:
            # LLM에 프롬프트를 전송하고 JSON 형식의 응답을 받음
            response = self.llm.invoke(prompt)
            # 응답에서 relevance_score 값을 추출
            score = json.loads(response.content)["relevance_score"]
            # 점수를 float로 변환하여 반환
            return float(score)
        except Exception as e:
            print(f"Error occurred: {str(e)}")
            return 5.0  # 에러 발생시 중간 점수로 처리하여 시스템 안정성 유지

    def postprocess_documents(self, documents: List[Document], query: str) -> List[Document]:
        # 벡터 검색으로 찾은 4개 문서를 LLM으로 재평가하여 최적의 2개 선택
        print('\n=== LLM이 4개의 검색 결과에 대해서 관련성을 평가합니다. ===')
        scored_docs = []
        for doc in documents:
            # 현재 처리 중인 문서에서 순수 텍스트 컨텐츠만 추출
            content = doc.page_content
            # LLM으로 문서 관련성 점수 계산 (1-10 사이 점수)
            score = self.evaluate_document(query, content)
            # 디버깅/모니터링을 위해 각 문서의 내용과 점수를 출력
            print(f"\nLLM 기반의 평가:\n{content}\n=> 점수: {score}\n")
            # 현재 문서와 계산된 점수를 튜플로 저장
            scored_docs.append((doc, score))

        # 모든 문서를 점수 기준 내림차순으로 정렬하고 상위 2개만 선택하여 반환
        ranked_docs = sorted(scored_docs, key=lambda x: x[1], reverse=True)
        return [doc for doc, _ in ranked_docs[:2]]


In [9]:
class SemanticRanker:
    # 벡터 검색 결과에 LLM 기반 의미적 평가를 적용하여 최적의 문서를 선별하는 시스템

    def __init__(self, vector_store, scorer):
        # 생성자에서 벡터 검색용 저장소와 LLM 기반 문서 평가기 인스턴스를 받아 저장
        self.vector_store = vector_store  # 벡터 검색용 저장소
        self.scorer = scorer  # LLM 기반 문서 평가기

    def retrieve(self, query: str) -> List[Document]:
        # 벡터 검색으로 유사도 기반 후보 문서 4개를 추출하고 LLM으로 재평가
        vector_results = self.vector_store.similarity_search(query, k=4)

        # 초기 벡터 검색 결과를 디버깅/분석용으로 출력
        print("\n=== 실제 검색 결과 (Top 4) ===")
        for i, doc in enumerate(vector_results, 1):
            print(f"\n검색 문서 {i}:")
            print(doc.page_content)

        # LLM으로 문서들을 재평가하고 재정렬하여 최적의 2개 선택
        reranked_results = self.scorer.postprocess_documents(vector_results, query)

        # 최종 선별된 문서를 디버깅/분석용으로 출력
        print("\n=== LLM의 리랭킹 결과 (Top 2) ===")
        for i, doc in enumerate(reranked_results, 1):
            print(f"\n검색 문서 {i}:")
            print(doc.page_content)

        return reranked_results

In [ ]:
# 문서 평가 및 검색 시스템 선언(초기화)
scorer = DocumentScorer(llm)  # LLM 기반 문서 평가기 생성
ranker = SemanticRanker(vector_store, scorer)  # 벡터 검색과 LLM 평가를 결합한 시스템 생성